# Churn Labeling (3-Rule Definition)

This notebook applies our churn definition to the billings data. We use a 3-rule system that differentiates between true churn, pre-churn (excluded), and successful renewals.

The three rules capture different churn scenarios:
1. **True Churn:** Customer didn't renew or renewed too late
2. **Pre-Churned:** Customer left before renewal window (different mechanism - excluded)
3. **Won:** Customer renewed successfully

## Setup

Importing config, helpers, and running the deduplication notebook to get clean data.

In [0]:
%run ../02_config/config_setup
%run ../utils/data_helpers
%run ./deduplication

## The 3-Rule Churn Definition

Here's how we label each customer:

| Rule | Condition | Label | Included in Model? |
|------|-----------|-------|-----------------|
| **Rule 1** | `Prospect_Outcome='Churned'` AND `Closed_Date is NULL` | Churned | ✅ Yes |
| **Rule 2** | `Prospect_Outcome='Churned'` AND `days_to_close > 28` | Churned | ✅ Yes |
| **Rule 3** | `Prospect_Outcome='Churned'` AND `days_to_close < 0` | Pre_Churned | ❌ No (excluded) |
| **Won** | `Prospect_Outcome='Won'` | Won | ✅ Yes |
| **Open** | `Prospect_Outcome='Open'` | Open | ❌ No (excluded) |

**Why 28 days?** A customer who closes more than 28 days after their renewal date has effectively lapsed. Late closure is operationally equivalent to non-renewal.

**Why exclude Pre-Churned?** These customers left *before* their renewal window opened. This is early cancellation or business closure - a fundamentally different mechanism from renewal failure.

In [0]:
section("Applying 3-rule churn definition")

# Define the three churn rules using numpy's select function
conditions = [
    # Rule 1: Never closed
    (billings['Prospect_Outcome'] == 'Churned') & (billings['Closed_Date'].isna()),
    
    # Rule 2: Closed too late (more than 28 days after renewal date)
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] > CHURN_DAYS_THRESHOLD),
    
    # Rule 3: Closed before renewal date (pre-churned, will be excluded)
    (billings['Prospect_Outcome'] == 'Churned') & (billings['days_to_close'] < 0),
    
    # Won: Successfully renewed
    billings['Prospect_Outcome'] == 'Won',
    
    # Open: Still in progress
    billings['Prospect_Outcome'] == 'Open',
]

choices = ['Churned', 'Churned', 'Pre_Churned', 'Won', 'Open']

# Apply the rules
billings['Churn_Label'] = np.select(conditions, choices, default='Other')

print("\nChurn label distribution:")
print(billings['Churn_Label'].value_counts().to_frame('Count'))

## Create Model DataFrame

For model training, we only keep **Won** and **Churned** records. We exclude:
- **Pre_Churned:** Different churn mechanism (early cancellation)
- **Open:** Still in progress, outcome unknown
- **Other:** Edge cases

In [0]:
section("Building model_df (Won + Churned only)")

# Keep only Won and Churned for modeling
model_df = billings[billings['Churn_Label'].isin(['Won', 'Churned'])].copy()

# Create binary target variable (1 = Churned, 0 = Won)
model_df['Target'] = (model_df['Churn_Label'] == 'Churned').astype(int)

# Add time-based features for analysis
model_df['Renewal_Year'] = model_df['Prospect_Renewal_Date'].dt.year
model_df['Renewal_Month'] = model_df['Prospect_Renewal_Date'].dt.to_period('M').astype(str)
model_df['Renewal_MonthNum'] = model_df['Prospect_Renewal_Date'].dt.month

print(f"  Total records (Won + Churned): {len(model_df):,}")
print(f"  Won        : {(model_df['Target']==0).sum():,}")
print(f"  Churned    : {(model_df['Target']==1).sum():,}")
print(f"  Churn rate : {model_df['Target'].mean()*100:.2f}%")

## Yearly Breakdown

Let's see how churn rates vary by year. This helps us understand trends and decide which year to use for training.

In [0]:
section("Churn rate by renewal year")

yr_summary = model_df.groupby('Renewal_Year').agg(
    Total=('Target', 'size'),
    Churned=('Target', 'sum')
).reset_index()

yr_summary['Churn_Rate'] = (yr_summary['Churned'] / yr_summary['Total'] * 100).round(2)

print("\nYearly churn summary:")
display_df(yr_summary, n=10)

print(f"\n  → 2023 has the highest churn rate ({yr_summary[yr_summary['Renewal_Year']==2023]['Churn_Rate'].values[0]:.2f}%)")
print(f"  → Using 2023 as training year captures peak churn behavior")

## Summary

Churn labels applied successfully. We now have a clean model_df ready for feature engineering.

In [0]:
print("\n" + "="*65)
print("  CHURN LABELING COMPLETE")
print("="*65)
print(f"\n  model_df created: {model_df.shape[0]:,} rows × {model_df.shape[1]} cols")
print(f"  Overall churn rate: {model_df['Target'].mean()*100:.2f}%")
print(f"  Training year (2023) churn rate: {model_df[model_df['Renewal_Year']==2023]['Target'].mean()*100:.2f}%")
print(f"\n  Next step: Feature engineering from multiple data sources")